# Lab 09 *(Self-Study, Architecture Reference)*: Network-Enforced Tool Governance with Agent Gateway
Building Enterprise AI Agents on **Gemini Enterprise Agent Platform (GEAP)**: Pulse Telco Workshop

---

## 🎯 Objectives
In Labs 01 to 06 you built, hardened, and deployed **Pulse Concierge** (version 5) to **Agent Runtime** with in-process guardrails (*ModelArmorPlugin*, *after_tool_callback*, and read-only MCP tool lists). When one central engineering team writes and reviews a single agent, application-level controls inside the Python container are sufficient. When an enterprise scales to **dozens of teams deploying autonomous agents across multiple frameworks**, relying solely on in-process code creates an architectural blind spot: an application team can misconfigure or omit a callback, or a compromised dependency can open outbound connections to unregistered hosts.

In this self-study architecture reference lab, you examine how GEAP enforces **network egress governance** outside the agent process using **Agent Gateway**:
1. Understand why scaling autonomous agents requires a regional default-deny proxy (*AGENT_TO_ANYWHERE*) bound directly to the **Agent Runtime** container network path.
2. Trace how the four gateway building blocks fit together: **Agent Registry** (destination catalog and MCP tool annotations), **Agent Gateway** (egress proxy and TLS termination), **IAP Authorization Extension** (*REQUEST_AUTHZ* policy decision point), and **Model Armor Extension** (*CONTENT_AUTHZ* wire payload screening).
3. Author and evaluate **IAM v3 Unified Access Policies (UAP)** with per-tool Common Expression Language (CEL) predicates (*destination.agent_registry.mcp_server.tool.annotations.read_only_hint == true*) and promote rules safely from *DRY_RUN* to *ENFORCED*.
4. Understand regional template coupling (*europe-west4*), request/response template direction inversion at the egress boundary, operational limits (one egress gateway per project), and how to deploy the complete stack in your own GCP environment using [reference-production-agent/terraform](../reference-production-agent/terraform).

> [!NOTE]
> **Why this lab is a self-study architecture reference:** Provisioning and propagating an Agent Gateway takes roughly 30 minutes, Agent Runtime allows only **one active egress gateway per project**, and binding per-tool IAM v3 access policies requires lifting organization-level policy constraints (*iam.managed.disableAccessPolicyBinding* and, for gateway Model Armor, *iam.allowedPolicyMemberDomains*) that temporary workshop accounts cannot modify. All Terraform code and validation scripts are provided in [reference-production-agent/terraform](../reference-production-agent/terraform) and [reference-production-agent/AGENT_GATEWAY_DEMO.md](../reference-production-agent/AGENT_GATEWAY_DEMO.md) so you can deploy and test the gateway in your own GCP organization.

### Key References & Official Documentation
* 🌐 [Agent Gateway overview](https://docs.cloud.google.com/gemini-enterprise-agent-platform/govern/gateways/agent-gateway-overview) · [Set up Agent Gateway](https://docs.cloud.google.com/gemini-enterprise-agent-platform/govern/gateways/set-up-agent-gateway)
* 🔐 [Configure IAM Unified Access Policies](https://docs.cloud.google.com/gemini-enterprise-agent-platform/govern/policies/configure-iam-policies-uap) · [CEL attributes for UAP](https://docs.cloud.google.com/gemini-enterprise-agent-platform/govern/policies/cel-attributes-uap)
* 🛡️ [Configure Model Armor on a gateway](https://docs.cloud.google.com/gemini-enterprise-agent-platform/govern/configure-model-armor) · [Monitor gateway traffic](https://docs.cloud.google.com/gemini-enterprise-agent-platform/govern/gateways/monitor-agent-gateway)
* 🗂️ [Agent Registry overview](https://docs.cloud.google.com/gemini-enterprise-agent-platform/govern/agent-registry) · 🪪 [Agent Identity overview](https://docs.cloud.google.com/gemini-enterprise-agent-platform/govern/agent-identity-overview)


## 💶 Cost and Scaling

| Service | How it is billed (list price, verified 2026-10-20) | This lab |
| :--- | :--- | :--- |
| Agent Gateway (*AGENT_TO_ANYWHERE* egress proxy in *europe-west4*) | Billed as Agent Compute: **1 vCPU-h (\$0.085) per 15,000 API calls or authorization requests** processed via Agent Gateway, i.e. ≈ **\$5.67 per 1M calls**, prorated hourly per project (effective 2026-07-13) | **\$0.00** (self-study architecture lab) |
| Model Armor on the gateway (*CONTENT_AUTHZ* in *europe-west4*) | **Free up to 2M tokens/month** per billing account, then **\$0.10 per 1M tokens** (same price in *europe-west4* and *eu*) | **\$0.00** |
| Agent Registry (*europe-west4*) & IAM v3 Unified Access Policy evaluation | **No separate public SKU** beyond the Agent Gateway per-request charge as of 2026-10-20 | **\$0.00** |
| Cloud Logging (*networkservices.googleapis.com/Gateway*) | **\$0.50/GiB** ingested after the first **50 GiB/project/month free** | **\$0.00** |

**How it scales:**
* **Registry capacity per gateway**: Each Agent Gateway instance governs up to **5,000 Agent Registry resources** and resolves destinations from the regional Agent Registry in its own region (*europe-west4*).
* **Call amplification per turn**: Because every outbound HTTPS connection from a gateway-bound Agent Runtime container traverses the proxy (Gemini model calls, telemetry export, Agent Registry lookups, MCP tool calls, and A2A delegations), a single conversational turn typically generates **10 to 25 outbound calls** through the gateway.
* **Single egress gateway per project**: Agent Runtime supports at most **one active *AGENT_TO_ANYWHERE* egress gateway per project**. Multiple agents deployed in the same project share that single gateway and isolate tool permissions by matching each agent's cryptographic SPIFFE *principal* in the IAM v3 UAP CEL rules.

**Estimated cost for this lab: \$0.00** (self-study architecture reference; no resources are provisioned in the workshop project).

Pricing references: [Gemini platform pricing](https://cloud.google.com/products/gemini-enterprise-agent-platform/pricing) · [Model Armor pricing](https://cloud.google.com/security/products/model-armor) · [Cloud Observability pricing](https://cloud.google.com/products/observability/pricing)


## 1. Why In-Process Guardrails Need a Network Egress Control Point

In Labs 03 to 06, *pulse_concierge* v5 enforced tool filters and Model Armor screening inside its own Python process, and could open outbound HTTPS connections to any URL known to its code (an MCP server, a remote A2A agent, or an arbitrary external host). When one central team writes and reviews a single agent, application-level controls (Lab 05) are sufficient. When an enterprise scales to **dozens of teams deploying autonomous agents across multiple frameworks**, relying solely on in-process code creates an architectural blind spot because an application team can misconfigure or omit a callback, or a compromised dependency can exfiltrate data to an external host. Security and platform architects need an out-of-process, network-level enforcement boundary that answers four questions independently of the agent's Python code:
* Which specific agent principal is allowed to call which MCP tool or remote A2A agent?
* Can an analytics agent invoke a mutating tool (*grant_dataset_access*), or only read-only tools (*readOnlyHint = true*)?
* Can an agent container open a connection to an unregistered external hostname?
* Where is the tamper-evident network audit log of every allowed and denied tool invocation?

**Agent Gateway** is a Google-managed, regional **default-deny proxy** for agent traffic ([Agent Gateway overview](https://docs.cloud.google.com/gemini-enterprise-agent-platform/govern/gateways/agent-gateway-overview)):
* **AGENT_TO_ANYWHERE (egress mode)**: When an Agent Runtime instance is bound to an egress gateway (*agents-cli deploy --agent-gateway-egress GATEWAY_URI*), every outbound connection from the agent container is routed through the gateway so container code cannot bypass it. (*CLIENT_TO_AGENT* is ingress mode; note that ingress mode is not supported for Gemini Enterprise apps.)
* **TLS termination and Root CA trust**: Because the gateway inspects encrypted MCP JSON-RPC methods and tool payloads over HTTPS (operating like a corporate TLS-inspection proxy), the agent container must trust the gateway's Root CA (*agents-cli scaffold enhance . --agent-gateway* adds the CA installation step to the Dockerfile, and *--build-args AGENT_GATEWAY_ROOT_CERTIFICATES=...* injects the certificate at build time).
* **Cryptographic SPIFFE Workload Identity**: The gateway automatically authenticates the calling container using its **Agent Identity** SPIFFE URI (formatted as "principal://agents.global.org-ORG_ID.system.id.goog/resources/aiplatform/projects/PROJECT_NUMBER/locations/REGION/reasoningEngines/ENGINE_ID"). The agent never manages API keys for the gateway.
* **Azure and OpenAI analogy**: Conceptually similar to combining **Azure API Management AI Gateway, Entra Agent ID conditional access, and Azure AI Content Safety**, with the critical difference that the egress proxy is bound directly to the Agent Runtime container network path.

```mermaid
sequenceDiagram
    autonumber
    participant Agent as 🎯 pulse_concierge<br/>(SPIFFE Agent Identity)
    participant GW as 🌐 Agent Gateway<br/>(AGENT_TO_ANYWHERE, europe-west4)
    participant Reg as 🗂️ Agent Registry<br/>(europe-west4)
    participant IAP as 🔐 IAP Authz Extension<br/>(REQUEST_AUTHZ + IAM v3 UAP CEL)
    participant MA as 🛡️ Model Armor Extension<br/>(CONTENT_AUTHZ, europe-west4)
    participant Tool as 🛠️ MCP Server / NOC A2A Agent

    Agent->>GW: Outbound HTTPS call (MCP tools/call or A2A message/send)
    GW->>Reg: Resolve destination host & MCP tool annotations (readOnlyHint)
    GW->>IAP: Evaluate IAM v3 Unified Access Policy (CEL)
    alt Policy denies (e.g. grant_dataset_access has read_only_hint == false)
        IAP-->>GW: DENIED (or logged as would-deny in DRY_RUN)
        GW-->>Agent: HTTP 403 Forbidden (logged to networkservices.googleapis.com/Gateway)
    else Policy allows
        IAP-->>GW: ALLOWED
        GW->>MA: Screen request payload (pulse-prod-guard-output)
        GW->>Tool: Forward request
        Tool-->>GW: Tool response (e.g. tok_audit_ext payload)
        GW->>MA: Screen response payload (pulse-prod-guard-input)
        MA-->>Agent: Deliver clean response (or HTTP 403 if Model Armor matched)
    end
```


## 2. How the Four Gateway Components Fit Together

| Component | GCP Resource Type | Architectural Role | In the Reference Terraform Module |
| :--- | :--- | :--- | :--- |
| **1. Catalog** | **Agent Registry** (ENDPOINT, MCP_SERVER, AGENT) in europe-west4 | Only **registered** destinations can be reached; supplies MCP tool annotations (*readOnlyHint*, *destructiveHint*) to the policy engine | Registers pulse-iam-context-mcp (with tool specs), pulse-network-ops-agent (A2A card), and the required Google API (.googleapis.com) endpoints |
| **2. Egress Proxy** | *networkservices.googleapis.com/AgentGateway* (*AGENT_TO_ANYWHERE*) | Intercepts all outbound traffic from bound Agent Runtime containers and writes network audit logs (*networkservices.googleapis.com/Gateway*) | *pulse-prod-egress-gateway* |
| **3. Access Control (*REQUEST_AUTHZ*)** | *AuthzExtension* (IAP) + *AuthzPolicy* + **IAM v3 Unified Access Policy (UAP)** | **IAP** acts as the policy decision point (evaluating CEL rules per tool call in **DRY_RUN** or **ENFORCED** mode, not as a browser login proxy) | pulse-prod-iap-authz-ext, pulse-prod-iap-authz-policy, and pulse-prod-egress-policy |
| **4. Wire Screening (*CONTENT_AUTHZ*)** | *AuthzExtension* (Model Armor) + *AuthzPolicy* | Screens MCP and A2A request and response payloads on the wire before they reach the tool or return to the LLM | pulse-prod-ma-content-ext and pulse-prod-ma-content-policy |

## 3. Per-Tool Governance with IAM v3 Unified Access Policies (UAP) and CEL
Unlike coarse project-level IAM v1 role bindings, Agent Gateway evaluates an **IAM v3 Unified Access Policy (UAP)** bound to the project ([Configure IAM policies for UAP](https://docs.cloud.google.com/gemini-enterprise-agent-platform/govern/policies/configure-iam-policies-uap), [CEL attributes reference](https://docs.cloud.google.com/gemini-enterprise-agent-platform/govern/policies/cel-attributes-uap)). Each rule pairs the calling agent's SPIFFE *principal* with a Common Expression Language (CEL) predicate over the destination and MCP method:

```text
# Rule 1: Essential platform endpoints (Gemini, telemetry, BigQuery/Knowledge Catalog MCP):
destination.agent_registry.endpoint.name in [
  '//agentregistry.googleapis.com/projects/.../locations/europe-west4/endpoints/aiplatform-eu', ...
]

# Rule 2: Registered A2A agents (allow-list specific agents by registry resource name):
destination.agent_registry.type == 'AGENT' &&
destination.agent_registry.agent.name == '//agentregistry.googleapis.com/projects/.../locations/europe-west4/agents/...'

# Rule 3: MCP protocol handshake and discovery (initialize, notifications/initialized, tools/list):
destination.agent_registry.type == 'MCP_SERVER' &&
request.mcp.method != 'tools/call'

# Rule 4: MCP tool execution: allow ONLY tools annotated with readOnlyHint = true:
destination.agent_registry.type == 'MCP_SERVER' &&
request.mcp.method == 'tools/call' &&
destination.agent_registry.mcp_server.tool.annotations.read_only_hint == true
```

* **Runtime mechanics on the wire**: When *pulse_concierge* invokes *resolve_user_context* or *check_bq_dataset_access* (where readOnlyHint is true), Rule 4 evaluates to **ALLOWED** and the gateway forwards the JSON-RPC call. If the model attempts to invoke *grant_dataset_access* (where readOnlyHint is false), the gateway rejects the request at the network edge with **HTTP 403 Forbidden** before the packet ever reaches the Cloud Run MCP server.
* **Safe rollout from DRY_RUN to ENFORCED**: Because default-deny proxies will break an agent if a required Google API endpoint is missing from the allow-list, you always start with *DRY_RUN* enforcement mode on the IAP *AuthzExtension*. In *DRY_RUN*, no traffic is blocked, and every tool-call log entry in *networkservices.googleapis.com/Gateway* records the decision IAP would have enforced. Once the gateway logs confirm all required endpoints are registered, flipping the extension to *ENFORCED* is a 1-minute metadata update requiring no container rebuild.


## 4. Wire-Level Model Armor (*CONTENT_AUTHZ*) vs In-Process ADK Plugin
When you attach a *CONTENT_AUTHZ* policy to the egress gateway ([Configure Model Armor on a gateway](https://docs.cloud.google.com/gemini-enterprise-agent-platform/govern/configure-model-armor)):
* **Regional template coupling (europe-west4)**: Unlike the in-process ADK *ModelArmorPlugin* in Lab 05 (which can call multi-region "eu" templates directly), **an Agent Gateway's *CONTENT_AUTHZ* templates must live in the exact same single region as the gateway (europe-west4)**.
* **Why request and response template directions invert at the egress boundary**: From the egress proxy's perspective, an outbound HTTP request leaving the container is the agent's output (*request_template_id = pulse-prod-guard-output*), whereas the HTTP response returning from an external MCP server or A2A agent is untrusted input entering the LLM context (*response_template_id = pulse-prod-guard-input*). When *resolve_user_context(tok_audit_ext)* returns the poisoned compliance note, the gateway's response template blocks it on the wire before the container sees the response.
* **Avoiding double-screening in production**: Running both the in-process ADK *ModelArmorPlugin* plus *after_tool_callback* (Lab 05) and gateway *CONTENT_AUTHZ* screens tool payloads twice, doubling Model Armor token consumption and adding 100 to 300 ms of callout latency. In production architectures, assign each layer a distinct boundary by using the in-process ADK plugin for direct user prompts and final model answers, and using gateway *CONTENT_AUTHZ* for external MCP and A2A traffic (note that Google API .googleapis.com traffic is automatically excluded from gateway *CONTENT_AUTHZ*).

## 5. Operational Caveats and Limits Before Adopting Agent Gateway
1. **Single egress gateway per project**: Agent Runtime allows **at most one active *AGENT_TO_ANYWHERE* (egress) Agent Gateway per project**. Attempting to bind an agent to a second egress gateway in the same project fails with HTTP 400 INVALID_ARGUMENT ("Another Agent Gateway is already active or being created for this project and direction"). Multiple agents in the same project share one egress gateway and isolate permissions by matching each agent's SPIFFE *principal* in the UAP CEL rules.
2. **Organization Policy prerequisites (*apply_org_policy_overrides*)**:
   * **constraints/iam.managed.disableAccessPolicyBinding** is enforced by default in new GCP organizations and blocks *gcloud iam policy-bindings create* for IAM v3 UAP policies. An **Organization Policy Administrator** must lift it on the project before *ENFORCED* mode can be bound.
   * **constraints/iam.allowedPolicyMemberDomains (Domain Restricted Sharing)** can block granting *roles/modelarmor.calloutUser*, *roles/modelarmor.user*, and *roles/serviceusage.serviceUsageConsumer* to the gateway's Google-managed service account (service-TENANT_NUMBER@gcp-sa-dep.iam.gserviceaccount.com), which resides in a Google tenant project outside your Workspace domain.
3. **Explicit hostname registration**: Because the gateway enforces default-deny matching on exact hostname and port, every Google API endpoint called by SDK client libraries (including regional europe-west4-aiplatform.googleapis.com, multi-region .rep.googleapis.com, and mtls. variants) must be registered as an *ENDPOINT* in Agent Registry. Always burn in new agents under *DRY_RUN* first to identify any unregistered endpoints in the gateway logs.
4. **Fail-closed availability trade-off**: Both the IAP and Model Armor gateway extensions operate with *failOpen=false* and a strict 1-second callout timeout (1s). If an extension callout times out or errors, the gateway denies the tool call.
5. **Per-tool CEL applies to MCP (*MCP_SERVER*), not A2A skills**: CEL rules can inspect *destination.agent_registry.mcp_server.tool.name* along with the read_only_hint, destructive_hint, idempotent_hint, and open_world_hint annotations. A2A calls are governed at the whole-agent level (*destination.agent_registry.agent.name*) rather than per individual A2A skill. When a mutating workflow requires deterministic human approval, enforce a human-in-the-loop gate in a graph workflow (covered in Lab 08).


## 6. Three-Stage Validation Flow and Self-Service Terraform Deployment
The reference module in [reference-production-agent/AGENT_GATEWAY_DEMO.md](../reference-production-agent/AGENT_GATEWAY_DEMO.md) validates Agent Gateway across three progressive stages:
1. **Stage 1 (*DRY_RUN*)**: Both *resolve_user_context* (readOnlyHint=true) and *grant_dataset_access* (readOnlyHint=false) succeed at runtime, while *networkservices.googleapis.com/Gateway* logs record the *DRY_RUN* policy verdict for each call.
2. **Stage 2 (*ENFORCED*)**: Flipping enforcement to *ENFORCED* allows *resolve_user_context* (HTTP 200, ALLOWED) and blocks *grant_dataset_access* at the network proxy (HTTP 403, DENIED).
3. **Stage 3 (*CONTENT_AUTHZ*)**: Calling *resolve_user_context* for "tok_audit_ext" passes IAP authorization, and its poisoned indirect-injection response is blocked on the wire by the gateway's Model Armor extension.

To deploy the complete reference architecture (including the optional Agent Gateway stack) in your own GCP project:
```bash
cd reference-production-agent/terraform
cp terraform.tfvars.example terraform.tfvars   # set project_id (and region = "europe-west4")
terraform init
# Deploy with Agent Gateway enabled in DRY_RUN mode:
terraform apply -var enable_agent_gateway=true -var agent_gateway_enforcement=DRY_RUN \
  -var agent_gateway_model_armor=true -var apply_org_policy_overrides=true
# Flip to ENFORCED (~2 min, no agent rebuild):
terraform apply -var enable_agent_gateway=true -var agent_gateway_enforcement=ENFORCED \
  -var agent_gateway_model_armor=true -var apply_org_policy_overrides=true
```
See [reference-production-agent/README.md](../reference-production-agent/README.md) and [reference-production-agent/terraform/variables.tf](../reference-production-agent/terraform/variables.tf) for full details on *enable_agent_gateway*, *agent_gateway_enforcement*, *agent_gateway_model_armor*, and *apply_org_policy_overrides*.


### 🧪 Try It Yourself (5 minutes): Write a CEL Rule for Destructive Tools and A2A Allow-Listing
Suppose the IAM Context MCP server adds a new tool *rotate_audit_keys* annotated with readOnlyHint=false and destructiveHint=true, and a second team registers an experimental A2A service *pulse-experimental-agent* in Agent Registry.
1. Using the [CEL attributes for UAP](https://docs.cloud.google.com/gemini-enterprise-agent-platform/govern/policies/cel-attributes-uap), write a CEL condition that allows MCP *tools/call* whenever *destination.agent_registry.mcp_server.tool.annotations.destructive_hint == false* (even if read_only_hint is false), **except** for *grant_dataset_access* by name (*destination.agent_registry.mcp_server.tool.name != "grant_dataset_access"*).
2. Compare Rule 2 in Section 3 above: why does matching *destination.agent_registry.agent.name* against the NOC agent's resource URI prevent *pulse_concierge* from calling *pulse-experimental-agent*, while still allowing *open_noc_dispatch_ticket* on the NOC agent?

## ✅ Lab 09 Complete: Architecture Summary

* **Out-of-process network governance (*AGENT_TO_ANYWHERE*)**: Binding an Agent Runtime instance to a regional Agent Gateway routes all outbound traffic through a default-deny proxy that authenticates the caller's cryptographic **SPIFFE Agent Identity** and blocks unregistered hostnames.
* **Per-tool CEL policies with safe rollout**: Combining **Agent Registry** tool specifications with **IAM v3 Unified Access Policies (UAP)** lets security teams enforce fine-grained predicates such as *read_only_hint == true* across MCP servers, burning in policies in *DRY_RUN* mode before switching to *ENFORCED* without rebuilding the agent container.
* **Wire-level Model Armor (*CONTENT_AUTHZ*) and operational boundaries**: Attaching regional Model Armor templates in *europe-west4* screens external MCP and A2A payloads on the wire, subject to the 1-egress-gateway-per-project limit and organization policy prerequisites.

👉 **Next:** **[Lab 10: Publishing Agents to Gemini Enterprise](lab10_gemini_enterprise.ipynb)**.
